In [5]:
# Install dataset package
!pip install ucimlrepo

# Import libraries
import pandas as pd
import numpy as np
import json
import hashlib
from datetime import datetime

from ucimlrepo import fetch_ucirepo
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, roc_auc_score, confusion_matrix

# Load UCI Heart Disease dataset
heart_disease = fetch_ucirepo(id=45)

# Features and target
X = heart_disease.data.features
y = heart_disease.data.targets

# Combine into one dataframe
df = pd.concat([X, y], axis=1)

# Display first few rows
df.head()

# Check dataset information
print(df.shape)
print(df.head())
print(df.isnull().sum())

# Drop rows with missing values for Version 1
df_clean = df.dropna().copy()

# The target column is usually called 'num'
# Convert target to binary:
# 0 = no heart disease
# 1 = heart disease present
df_clean["heart_disease_present"] = df_clean["num"].apply(lambda x: 1 if x > 0 else 0)

# Separate features and target
X = df_clean.drop(columns=["num", "heart_disease_present"])
y = df_clean["heart_disease_present"]

print(X.shape)
print(y.value_counts())

# Split data into training and testing sets
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

# Create logistic regression model
model = LogisticRegression(max_iter=1000)

# Train model
model.fit(X_train, y_train)

# Make predictions
y_pred = model.predict(X_test)
y_prob = model.predict_proba(X_test)[:, 1]

# Evaluate model
metrics = {
    "accuracy": float(accuracy_score(y_test, y_pred)),
    "precision": float(precision_score(y_test, y_pred)),
    "recall": float(recall_score(y_test, y_pred)),
    "f1_score": float(f1_score(y_test, y_pred)),
    "roc_auc": float(roc_auc_score(y_test, y_prob))
}

print(metrics)

# Save cleaned dataset as CSV for hashing
dataset_filename = "heart_disease_clean_v1.csv"
df_clean.to_csv(dataset_filename, index=False)

# Function to create file hash
def hash_file(filename):
    sha256_hash = hashlib.sha256()
    with open(filename, "rb") as f:
        for byte_block in iter(lambda: f.read(4096), b""):
            sha256_hash.update(byte_block)
    return sha256_hash.hexdigest()

# Function to create block hash
def hash_block(block):
    block_string = json.dumps(block, sort_keys=True).encode()
    return hashlib.sha256(block_string).hexdigest()

# Create audit block
audit_block_v1 = {
    "project_title": "Designing a Blockchain-Inspired Audit Trail for Machine Learning Models Used in Heart Disease Risk Prediction",
    "dataset_name": "UCI Heart Disease Dataset",
    "dataset_version": "v1_cleaned",
    "dataset_hash": hash_file(dataset_filename),
    "model_name": "Logistic Regression",
    "model_version": "v1",
    "training_date": datetime.now().isoformat(),
    "features_used": list(X.columns),
    "performance_metrics": metrics,
    "previous_block_hash": "0"
}

# Add current block hash
audit_block_v1["current_block_hash"] = hash_block(audit_block_v1)

# Display audit block
audit_block_v1

# Save audit trail record as JSON
with open("audit_trail_v1.json", "w") as f:
    json.dump(audit_block_v1, f, indent=4)

print("Audit trail saved as audit_trail_v1.json")



(303, 14)
   age  sex  cp  trestbps  chol  fbs  restecg  thalach  exang  oldpeak  slope  \
0   63    1   1       145   233    1        2      150      0      2.3      3   
1   67    1   4       160   286    0        2      108      1      1.5      2   
2   67    1   4       120   229    0        2      129      1      2.6      2   
3   37    1   3       130   250    0        0      187      0      3.5      3   
4   41    0   2       130   204    0        2      172      0      1.4      1   

    ca  thal  num  
0  0.0   6.0    0  
1  3.0   3.0    2  
2  2.0   7.0    1  
3  0.0   3.0    0  
4  0.0   3.0    0  
age         0
sex         0
cp          0
trestbps    0
chol        0
fbs         0
restecg     0
thalach     0
exang       0
oldpeak     0
slope       0
ca          4
thal        2
num         0
dtype: int64
(297, 13)
heart_disease_present
0    160
1    137
Name: count, dtype: int64
{'accuracy': 0.8333333333333334, 'precision': 0.8461538461538461, 'recall': 0.7857142857142857, 'f